# Data Quality Audit of Raw Data from the Joint Allocation Office (JAO)

The main objective of the data quality audit is to assess the quality, consistency, and general characteristics of the raw data obtained from the Joint Allocation Office (JAO).

The analysis focuses on the following aspects:
* Missing data and missing observations
* Index integrity and consistency
* Duplicated indices
* Timestamp and time zone consistency
* Overall data characteristics

**All data and variables fetched from JAO are explained in the [JAO data overview](./01_JAO_data_overview.ipynb).**

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent.parent.parent))
from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.data_quality import (
    print_index_summary,
    print_missing_values,
    find_missing_dates,
)

from src.utils.file_io import load_jao_data

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
data_jao = load_jao_data(DATA_DIR / "raw" / "jao" / date_range)

## 1. JAO data

### 1.1 Missing Data

In [3]:
for name, data in data_jao.items():
    print_missing_values(data=data, name=name)


--- [refprog] Missing values ---
border_DK1_UK_Viking1      30527
border_DK1_UK_Viking2      30527
border_IT_MEMONITA2         9528
border_ME_ITMONITA2         9528
border_RO_MD               27696
border_RS_KS                2016
border_UA_MD               27696
border_ES_MA_ESMA_link1    10368
border_ES_MA_ESMA_link2    10368
border_KS_ME               10368
border_SI_HU               10368
border_KS_AL               10368
border_MK_KS               10368
border_DE_DE_DK1_VH        30839
border_DE_DK2_BigHub_DE    30839
border_NL_DK1_COBRA        30839
border_RO_RO_BG_VH         30839
border_NL_NL_NO2_NorNed    30839
border_DE_NO2_BigHub_DE    30839
border_DE_DE_SE4_Baltic    30839
border_PL_PL_SE4_SwePol    30839
border_PL_LT_BigHub_PL     30839

--- [minmax_np] Missing values ---
minDE_NO2_BigHub    30911
minNL_NO2_NorNed    30911
minDE_DK2_BigHub    30911
minDE_SE4_Baltic    30911
minPL_LT_BigHub     30911
minPL_SE4_SwePol    30911
minRO_BG_VH         30911
minNL_DK1_COBRA     30

### 1.2 Missing Data – Observations

Missing values occur only in `refprog` (22 border columns), `minmax_np` (18 min/max columns) and `allocation_constraint` (`BE_import`, `BE_export`). `d2cf` and all remaining columns are complete. The gaps are structural (a series not published for a given link over most of the sample), not random loss.

Nine non-Core links, including the Polish interconnectors with Lithuania (`PL-LT`) and Sweden (`PL-SE4`), are missing 30,839 of 33,167 values in `refprog` and 30,911 of 33,263 in `minmax_np`. Their data is available only from June 2026, which is related to the introduction of the Core Advanced Hybrid Coupling (AHC) process on 11 June 2026. The other affected `refprog` borders (non-Core or Core–non-Core links) are missing between 2,016 and 30,527 values. In `allocation_constraint`, `BE_export` is empty in all 58,391 rows and `BE_import` is missing 47,903 values.

Even the smallest gap (2,016 values) equals 84 days of hourly data, so none of the affected columns can be reliably imputed.

### 1.3 Index Integrity

In [4]:
for name, data in data_jao.items():
    print_index_summary(data=data, name=name)


--- [refprog] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 3
Timezone:   Europe/Amsterdam

Time steps:
  0 days 01:00:00: 33163
  1 days 01:00:00: 3

--- [minmax_np] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 3
Timezone:   Europe/Amsterdam

Time steps:
  0 days 01:00:00: 33262

--- [d2cf] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 3
Timezone:   Europe/Amsterdam

Time steps:
  0 days 01:00:00: 33155
  0 days 02:00:00: 4
  1 days 01:00:00: 3

--- [allocation_constraint] Index summary ---
Duplicates: 0
Duplicates due to Summer/Winter time switch: 6
Timezone:   Europe/Amsterdam

Time steps:
  0 days 00:15:00: 33599
  0 days 01:00:00: 24789
  1 days 01:00:00: 2


In [5]:
for name, data in data_jao.items():
    find_missing_dates(data=data, name=name)


--- [refprog] Missing timestamps (freq=h) ---
Count: 72

Missing ranges:
  2023-10-30 00:00:00+01:00 - 2023-10-30 23:00:00+01:00 (24 timestamps)
  2024-06-25 00:00:00+02:00 - 2024-06-25 23:00:00+02:00 (24 timestamps)
  2024-10-28 00:00:00+01:00 - 2024-10-28 23:00:00+01:00 (24 timestamps)

Missing timestamps by day of week:
  Monday: 48
  Tuesday: 24

--- [minmax_np] Missing timestamps (freq=h) ---
Count: 0

--- [d2cf] Missing timestamps (freq=h) ---
Count: 76

Missing ranges:
  2022-12-22 17:00:00+01:00 (1 timestamp)
  2022-12-25 23:00:00+01:00 (1 timestamp)
  2023-03-31 14:00:00+02:00 (1 timestamp)
  2023-10-30 00:00:00+01:00 - 2023-10-30 23:00:00+01:00 (24 timestamps)
  2024-05-05 08:00:00+02:00 (1 timestamp)
  2024-06-25 00:00:00+02:00 - 2024-06-25 23:00:00+02:00 (24 timestamps)
  2024-10-28 00:00:00+01:00 - 2024-10-28 23:00:00+01:00 (24 timestamps)

Missing timestamps by day of week:
  Monday: 48
  Tuesday: 24
  Thursday: 1
  Friday: 1
  Sunday: 2

--- [allocation_constraint] Miss

### 1.4 Index Integrity – Observations

All JAO series are tz-aware (Europe/Amsterdam) and contain no duplicated indices; the repeated timestamps come only from the autumn DST switch and are valid distinct instants, not duplicates. Resolution is hourly in `refprog`, `minmax_np` and `d2cf`, whereas `allocation_constraint` mixes 15-min and hourly steps, so a common grid is needed before merging.

`minmax_np` has no gaps. The other datasets contain two types of gaps:
* **Whole-day gaps (24 hourly MTUs)** – 2023-10-30, 2024-06-25 and 2024-10-28 in `refprog` and `d2cf`; 2023-10-30 and 2024-10-28 in `allocation_constraint` (in its hourly part). The two Mondays are the days right after the autumn DST switch and appear in all three datasets, which points to a systematic cause (publication) rather than random loss.
* **Isolated single-hour gaps** – 4 in `d2cf` (2022-12-22 17:00, 2022-12-25 23:00, 2023-03-31 14:00, 2024-05-05 08:00).

## 2. Conclusions and Data Cleaning / Preprocessing Decisions

**Main findings**
Missing values are confined to non-Core link columns in `refprog` and `minmax_np` and to `BE_import`/`BE_export` in `allocation_constraint`. Each of these columns is missing at least 2,016 values (84 days of hourly data), so they are dropped rather than imputed; `d2cf` and all other columns are complete. The time index has no duplicates, but contains whole-day gaps (24 hourly MTUs) in `refprog`, `d2cf` and `allocation_constraint` and 4 isolated single-hour gaps in `d2cf`; `minmax_np` is gap-free. `allocation_constraint` has mixed resolution (15-min and hourly).

### 2.1 Data Cleaning Decisions

**Columns to drop**

* `refprog` – all 22 columns with missing values (2,016–30,839 missing out of 33,167 rows): `border_DE_DE_DK1_VH`, `border_DE_DK2_BigHub_DE`, `border_NL_DK1_COBRA`, `border_RO_RO_BG_VH`, `border_NL_NL_NO2_NorNed`, `border_DE_NO2_BigHub_DE`, `border_DE_DE_SE4_Baltic`, `border_PL_PL_SE4_SwePol`, `border_PL_LT_BigHub_PL`, `border_DK1_UK_Viking1`, `border_DK1_UK_Viking2`, `border_RO_MD`, `border_UA_MD`, `border_ES_MA_ESMA_link1`, `border_ES_MA_ESMA_link2`, `border_KS_ME`, `border_SI_HU`, `border_KS_AL`, `border_MK_KS`, `border_IT_MEMONITA2`, `border_ME_ITMONITA2`, `border_RS_KS`.
* `minmax_np` – all 18 columns with missing values (30,911 missing out of 33,263 rows): `min` and `max` columns of `DE_NO2_BigHub`, `NL_NO2_NorNed`, `DE_DK2_BigHub`, `DE_SE4_Baltic`, `PL_LT_BigHub`, `PL_SE4_SwePol`, `RO_BG_VH`, `NL_DK1_COBRA` and `DE_DK1_VH` (e.g. `minPL_LT_BigHub`, `maxPL_LT_BigHub`).
* `allocation_constraint` – `BE_export` (58,391 missing out of 58,391 rows) and `BE_import` (47,903 missing out of 58,391 rows).
* `d2cf` – no columns to drop.

After dropping these columns no missing values remain, so no value imputation is needed. This also removes the Polish `PL-LT` and `PL-SE4` connections.

### 2.2 Data Preprocessing Decisions

**Time Index**

* Convert all indices to UTC before merging or comparing the datasets.
* Bring `allocation_constraint` to a common time grid with the other datasets (mixed 15-min and hourly steps).
* Reindex `refprog`, `minmax_np` and `d2cf` to a complete hourly grid before filling the gaps.

**Missing timestamps**

* `minmax_np` – no gaps, nothing to fill.
* Gaps of up to 3 hours (4 isolated single-hour gaps in `d2cf`) – linear interpolation.
* Longer gaps (all are whole-day gaps) – fill with values from the same hours one week earlier (D-7):
  * `refprog` and `d2cf`: 2023-10-30, 2024-06-25, 2024-10-28,
  * `allocation_constraint`: 2023-10-30, 2024-10-28.
* D-7 is used because JAO series (exchange programs, net positions, grid-model load) follow a weekly demand and market cycle, and most of the missing days are Mondays, whose previous day is a Sunday with a different profile (for 2023-10-30 and 2024-10-28 also a 25-hour DST day). The hours are matched by local time of day. Interpolation over 24 hours would flatten the daily profile, so it is limited to gaps of up to 3 hours (the same limit as in the ENTSO-E notebook).